# 🧠 Pipeline MLOps com Weights & Biases e MLP

Este notebook implementa um fluxo completo de Machine Learning utilizando conceitos de MLOps e a plataforma **Weights & Biases (W&B)**. O objetivo é criar um pipeline reprodutível e monitorado que vai desde o carregamento dos dados até o treinamento de uma **Multi‑Layer Perceptron (MLP)** com PyTorch.

Ao final, você terá:
* Versionamento de artefatos (dados brutos, limpos, splits, modelos) no W&B
* Testes automatizados de consistência dos dados
* Divisão treino/teste com validação estatística
* Modelo MLP com logging de métricas e early stopping
* Registro do melhor modelo como artefato

---

In [11]:
!pip install -r requirements.txt

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\Users\josem\OneDrive\Documentos\IA\.venv\Scripts\pip.exe\__main__.py", line 4, in <module>
    from pip._internal.cli.main import main
ModuleNotFoundError: No module named 'pip'


  Using cached wandb-0.25.1-py3-none-win_amd64.whl.metadata (11 kB)
  Using cached pandas-3.0.2-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.4.4-cp313-cp313-win_amd64.whl.metadata (6.6 kB)
  Using cached torch-2.11.0-cp313-cp313-win_amd64.whl.metadata (29 kB)
  Using cached scikit_learn-1.8.0-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached matplotlib-3.10.8-cp313-cp313-win_amd64.whl.metadata (52 kB)
  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
  Using cached click-8.3.1-py3-none-any.whl.metadata (2.6 kB)
  Using cached gitpython-3.1.46-py3-none-any.whl.metadata (13 kB)
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached requests-2.33.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached sentry_

ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\Users\\josem\\OneDrive\\Documentos\\IA\\.venv\\Lib\\site-packages\\scikit_learn-1.8.0.dist-info\\INSTALLER7v9zcbu5.tmp'



In [2]:
import wandb
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from scipy.stats import ks_2samp, chi2_contingency
import yaml
import random
import matplotlib.pyplot as plt
import kagglehub
import shutil
import os

# Configurar semente para reprodutibilidade
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

c:\Users\josem\OneDrive\Documentos\mlops-wandb-project\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [1]:
import wandb

API_KEY = "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" # Replace with your actual key
wandb.login(key=API_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\josem\_netrc
wandb: Currently logged in as: moiseslopesdasilva708811 (moiseslopesdasilva708811-ufrn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [7]:
import os
os.environ["WANDB__SERVICE_WAIT"] = "true"
os.environ["WANDB_DISABLE_SERVICE"] = "true"

In [ ]:
config = {
    "data": {
        "raw_path": "data/raw/dataset.csv",
        "test_size": 0.2,
        "random_state": 42,
        "target_col": "target",
        "missing_threshold": 0.5,
        "imputation_strategy": "mean"
    },
    "model": {
        "hidden_sizes": [64, 32],
        "dropout": 0.2,
        "learning_rate": 0.001,
        "batch_size": 32,
        "epochs": 100,
        "early_stopping_patience": 10
    }
}

In [ ]:
np.random.seed(42)
n_samples = 10000
n_features = 20

X = np.random.randn(n_samples, n_features)
X[:, 5:10] = np.random.exponential(scale=1, size=(n_samples, 5))

coef = np.random.randn(n_features)
logits = X @ coef + 0.5 * np.random.randn(n_samples)
y = (1 / (1 + np.exp(-logits)) > 0.5).astype(int)

X_missing = X.copy()
for col in [2, 7, 12]:
    missing_idx = np.random.choice(n_samples, size=int(0.05*n_samples), replace=False)
    X_missing[missing_idx, col] = np.nan

dup_idx = np.random.choice(n_samples, size=int(0.01*n_samples), replace=False)
X_missing = np.vstack([X_missing, X_missing[dup_idx]])
y = np.hstack([y, y[dup_idx]])

columns = [f'feature_{i}' for i in range(n_features)]
df_raw = pd.DataFrame(X_missing, columns=columns)
df_raw['target'] = y

print(f"Dataset gerado: {len(df_raw)} amostras, {len(df_raw.columns)} colunas")
df_raw.head()

In [8]:
wandb.init(project="mlops-wandb-project", job_type="load_raw", name="load_raw")
artifact = wandb.Artifact("raw_data", type="dataset")
temp_path = "temp_raw.csv"
df_raw.to_csv(temp_path, index=False)
artifact.add_file(temp_path)
wandb.log_artifact(artifact)
wandb.summary["rows"] = len(df_raw)
wandb.summary["columns"] = list(df_raw.columns)
wandb.finish()
print("Artefato raw_data salvo no W&B")

ServicePollForTokenError: Failed to read port info after 30.0 seconds.

In [ ]:
def remove_duplicates(df):
    before = len(df)
    df = df.drop_duplicates()
    print(f"Removed {before - len(df)} duplicates")
    return df

def handle_missing_values(df, strategy='mean', threshold=0.5):
    missing_frac = df.isnull().mean()
    cols_to_drop = missing_frac[missing_frac > threshold].index.tolist()
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped columns: {cols_to_drop}")
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    imputer = SimpleImputer(strategy=strategy)
    df[numeric_cols] = imputer.fit_transform(df[numeric_cols])
    cat_cols = df.select_dtypes(include=['object']).columns
    df[cat_cols] = df[cat_cols].fillna('missing')
    return df

df_clean = remove_duplicates(df_raw)
df_clean = handle_missing_values(df_clean, 
                                 strategy=config['data']['imputation_strategy'],
                                 threshold=config['data']['missing_threshold'])
print(f"Dataset limpo: {len(df_clean)} amostras")

wandb.init(project="mlops-wandb-demo", job_type="clean_data", name="clean_data")
artifact = wandb.Artifact("clean_data", type="dataset")
temp_path = "temp_clean.csv"
df_clean.to_csv(temp_path, index=False)
artifact.add_file(temp_path)
wandb.log_artifact(artifact)
wandb.summary["rows"] = len(df_clean)
wandb.finish()

In [ ]:
def test_no_missing_values(df):
    missing = df.isnull().sum().sum()
    assert missing == 0, f"Missing values found: {missing}"

def test_target_values(df, target_col, allowed_values):
    invalid = ~df[target_col].isin(allowed_values)
    assert invalid.sum() == 0, f"Invalid values in {target_col}: {df.loc[invalid, target_col].unique()}"

test_no_missing_values(df_clean)
test_target_values(df_clean, config['data']['target_col'], [0, 1])
print("Todos os testes de consistência passaram.")

In [ ]:
def split_train_test(df, target_col, test_size=0.2, random_state=42):
    X = df.drop(columns=[target_col])
    y = df[target_col]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    train_df = pd.concat([X_train, y_train], axis=1)
    test_df = pd.concat([X_test, y_test], axis=1)
    return train_df, test_df

def compare_distributions(train_df, test_df, columns):
    results = {}
    for col in columns:
        train_vals = train_df[col].dropna()
        test_vals = test_df[col].dropna()
        if train_df[col].dtype in ['int64', 'float64']:
            ks_stat, p_value = ks_2samp(train_vals, test_vals)
            results[col] = {'test': 'KS', 'statistic': ks_stat, 'p_value': p_value}
        else:
            train_counts = train_vals.value_counts(normalize=True)
            test_counts = test_vals.value_counts(normalize=True)
            all_cats = sorted(set(train_counts.index).union(set(test_counts.index)))
            train_probs = [train_counts.get(cat, 0) for cat in all_cats]
            test_probs = [test_counts.get(cat, 0) for cat in all_cats]
            chi2, p_value, _, _ = chi2_contingency([train_probs, test_probs])
            results[col] = {'test': 'Chi2', 'statistic': chi2, 'p_value': p_value}
    return results

train_df, test_df = split_train_test(df_clean, 
                                     target_col=config['data']['target_col'],
                                     test_size=config['data']['test_size'],
                                     random_state=config['data']['random_state'])

print(f"Treino: {len(train_df)} amostras")
print(f"Teste:  {len(test_df)} amostras")

feature_cols = [c for c in train_df.columns if c != config['data']['target_col']]
comp_results = compare_distributions(train_df, test_df, feature_cols)

wandb.init(project="mlops-wandb-demo", job_type="split_data", name="split_data")

train_artifact = wandb.Artifact("train_data", type="dataset")
train_artifact.add_file(wandb.save("temp_train.csv", df=train_df.to_csv(index=False)))
wandb.log_artifact(train_artifact)

test_artifact = wandb.Artifact("test_data", type="dataset")
test_artifact.add_file(wandb.save("temp_test.csv", df=test_df.to_csv(index=False)))
wandb.log_artifact(test_artifact)

comp_df = pd.DataFrame(comp_results).T
comp_table = wandb.Table(dataframe=comp_df)
wandb.log({"distribution_comparison": comp_table})

wandb.summary["train_size"] = len(train_df)
wandb.summary["test_size"] = len(test_df)

wandb.finish()
print("Artefatos de treino e teste salvos com sucesso.")

## 8. Modelo MLP: Definição e Matemática

### 🧠 Explicação Matemática da MLP

Uma **Multi‑Layer Perceptron (MLP)** é uma rede neural feedforward composta por:
* **Camada de entrada**: recebe os vetores de características.
* **Camadas ocultas**: cada neurônio aplica uma transformação afim seguida de uma função de ativação não linear.
* **Camada de saída**: produz a previsão final.

Para uma camada $l$ com $n$ neurônios:
- **Entrada**: $\mathbf{x}^{(l)} \in \mathbb{R}^{n_{in}}$
- **Pesos**: $\mathbf{W}^{(l)} \in \mathbb{R}^{n_{out} \times n_{in}}$
- **Bias**: $\mathbf{b}^{(l)} \in \mathbb{R}^{n_{out}}$
- **Ativação**: $\mathbf{z}^{(l)} = \mathbf{W}^{(l)} \mathbf{x}^{(l)} + \mathbf{b}^{(l)}$  
  $\mathbf{a}^{(l)} = f(\mathbf{z}^{(l)})$  (ex.: ReLU)

Na **classificação binária**, a camada de saída usa a função sigmoide para produzir uma probabilidade:

$$\hat{y} = \sigma(\mathbf{W}^{(L)} \mathbf{a}^{(L-1)} + \mathbf{b}^{(L)})$$

O treinamento minimiza a **entropia cruzada binária**:

$$\mathcal{L} = -\frac{1}{N}\sum_{i=1}^N \left[ y_i \log(\hat{y}_i) + (1-y_i)\log(1-\hat{y}_i) \right]$$

A atualização dos pesos é feita via **backpropagation** com **Adam** (variante do gradiente descendente).

### Implementação em PyTorch

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_sizes, output_dim=1, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

In [ ]:
def prepare_dataloaders(train_df, test_df, target_col, batch_size):
    X_train = train_df.drop(columns=[target_col]).values.astype(np.float32)
    y_train = train_df[target_col].values.astype(np.float32).reshape(-1, 1)
    X_test = test_df.drop(columns=[target_col]).values.astype(np.float32)
    y_test = test_df[target_col].values.astype(np.float32).reshape(-1, 1)
    
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_dataset = TensorDataset(torch.tensor(X_test), torch.tensor(y_test))
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    return train_loader, test_loader, scaler

def train_model(config, train_loader, test_loader, input_dim):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = MLP(input_dim, config['model']['hidden_sizes'], dropout=config['model']['dropout']).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=config['model']['learning_rate'])
    
    best_loss = float('inf')
    patience = config['model']['early_stopping_patience']
    counter = 0

    wandb.watch(model, log="all")

    for epoch in range(config['model']['epochs']):
        model.train()
        train_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * X_batch.size(0)
        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        correct = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                output = model(X_batch)
                loss = criterion(output, y_batch)
                val_loss += loss.item() * X_batch.size(0)
                pred = (torch.sigmoid(output) > 0.5).float()
                correct += (pred == y_batch).sum().item()
        val_loss /= len(test_loader.dataset)
        acc = correct / len(test_loader.dataset)

        wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss, "val_acc": acc})

        if val_loss < best_loss:
            best_loss = val_loss
            counter = 0
            torch.save(model.state_dict(), "best_model.pt")
            model_artifact = wandb.Artifact("trained_model", type="model")
            model_artifact.add_file("best_model.pt")
            wandb.log_artifact(model_artifact)
        else:
            counter += 1
            if counter >= patience:
                print(f"Early stopping triggered at epoch {epoch}")
                break

    model.load_state_dict(torch.load("best_model.pt"))
    return model

wandb.init(project="mlops-wandb-demo", job_type="train", name="mlp_training", config=config)

train_df = pd.read_csv("temp_train.csv")
test_df = pd.read_csv("temp_test.csv")

train_loader, test_loader, scaler = prepare_dataloaders(
    train_df, test_df, config['data']['target_col'], config['model']['batch_size']
)
input_dim = train_loader.dataset.tensors[0].shape[1]

model = train_model(config, train_loader, test_loader, input_dim)

wandb.finish()
print("Treinamento concluído e melhor modelo salvo no W&B.")

## 10. Conclusão

Parabéns! Você completou o pipeline MLOps com Weights & Biases. Neste notebook, você:
* Gerou um dataset sintético
* Versionou dados brutos e limpos como artefatos
* Aplicou testes automatizados
* Dividiu os dados mantendo a distribuição estatística
* Definiu e treinou uma MLP com PyTorch
* Registrou métricas e o melhor modelo no W&B

Agora você pode explorar os artefatos e logs no [app.wandb.ai](https://app.wandb.ai) e adaptar este fluxo para seus próprios projetos.

---
**Próximos passos sugeridos:**
* Substituir o dataset sintético pelo seu CSV
* Adicionar mais testes (ex.: validação de tipos, ranges)
* Experimentar com diferentes arquiteturas de modelo
* Integrar o pipeline em um sistema CI/CD

Divirta-se explorando MLOps com W&B! 🚀